## H1N1-VACCINE-ANALYSIS-PREDICTIONS

### Stakeholder
 #### Public Health Authorities

## Problem Statement
Public health authorities need to increase vaccination uptake to reduce the spread of infectious diseases and protect vulnerable population.
However, many individuals choose not to receive vaccines due to differences in beliefs, health behaviors, risk perception and access to healthcare.
Without understanding which groups are less likely to vaccinate, it is diicult for health organisation to design effective outreach and vaccination campaigns

### Business Understanding

## KEY QUESTION FOR PREDICTING H1N1 VACCINE UPTAKE


1.Which factors influence whether an individual receives the H1N1 flu vaccine and can we accurately predict who is likely or unilikely to get vaccinated?

2 Which demographic group are les likely to vaccinate? Understanding differences in vaccination behavior across age, gender, education level and income helps identify high risk populations.


3.Does doctor recommendation influences vaccination behavior? Medical professionals are trusted sources of health advice. Analysing whether individuals who receives recommendations from their doctors are more likely to vaccinate can guide strategies for improving outreach through healthcare providers.

4,Which health condition are associated with vaccination? People with chronic medical condition may be more motivated to vaccinate. identifying
these patterns helps príoritize interventions for high risk groups.

### Import Libraries
Importing libraries for data manupulation,visualization,machine learning, preprocessing and evalution.

In [33]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning models
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

## Load the Dataset
Load the dataset into a DataFrame and show the first 5 rows so that we can inspect the data

In [34]:
df = pd.read_csv("H1N1_Flu_vaccines.csv")
df.head()

,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,NaN,NaN,0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,NaN,NaN,0,1
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0


## Check Columns
This helps us see all the variables in the dataset and identfy the target columns, feature columns and unnecessary columns

In [35]:
print(df.columns.tolist())

['respondent_id', 'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'age_group', 'education', 'race', 'sex', 'income_poverty', 'marital_status', 'rent_or_own', 'employment_status', 'hhs_geo_region', 'census_msa', 'household_adults', 'household_children', 'employment_industry', 'employment_occupation', 'h1n1_vaccine', 'seasonal_vaccine']


## Check Dataset Shape and Missing Vlues
This is important because missing values must be handled before modeling

In [36]:
print(df.shape)
print(df.isnull().sum())

(26707, 38)
respondent_id                      0
h1n1_concern                      92
h1n1_knowledge                   116
behavioral_antiviral_meds         71
behavioral_avoidance             208
behavioral_face_mask              19
behavioral_wash_hands             42
behavioral_large_gatherings       87
behavioral_outside_home           82
behavioral_touch_face            128
doctor_recc_h1n1                2160
doctor_recc_seasonal            2160
chronic_med_condition            971
child_under_6_months             820
health_worker                    804
health_insurance               12274
opinion_h1n1_vacc_effective      391
opinion_h1n1_risk                388
opinion_h1n1_sick_from_vacc      395
opinion_seas_vacc_effective      462
opinion_seas_risk                514
opinion_seas_sick_from_vacc      537
age_group                          0
education                       1407
race                               0
sex                                0
income_poverty            

## Choose the Target Variable
This project focuses on predicting whether a person received the H1N1 Vaccine. Target Variable would be h1n1_vaccine and we will remove seasonal_vaccine  because it is another target variable and keeping it may cause data leakage

In [37]:
#Choose H1N1 vaccine as target
target = "h1n1_vaccine"

#features and target
X = df.drop(columns=["h1n1_vaccine","seasonal_vaccine"])
y = df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (26707, 36)
Target shape: (26707,)


## Train-Test Split
Split the dataset into training data (80%) used to train the model and testing data (20%) used to evaluate the model on unseen data. I used 'stratify=y' to keep same proportion of vaccinated and non-vaccinated individuals in both sets.

In [38]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (21365, 36)
Testing set: (5342, 36)
